# Advanced BDM Analysis

## Part of the open_dvm Toolbox

This tutorial builds on `05_bdm_decoding.ipynb` to explore advanced multivariate decoding techniques: temporal generalization, classifier/dimensionality-reduction comparisons, time-frequency decoding, and group-level permutation testing.

## Learning Objectives

After completing this tutorial, you will:

- **Assess temporal generalization** — Test whether a decoder trained at one time point generalizes to other time points (GAT)
- **Compare classifiers and PCA preprocessing** — See how classifier choice and dimensionality reduction affect decoding performance
- **Decode from time-frequency representations** — Identify which frequency bands carry decodable information
- **Establish group-level statistical significance** — Use permutation testing across subjects

**Prerequisites:** This tutorial assumes familiarity with the concepts introduced in `05_bdm_decoding.ipynb`.

## Overview

### Key Steps
1. **Generalization Across Time (GAT)**: Train at each time point, test at all others
2. **Classifier comparisons with PCA**: Compare LDA, SVM, and Gaussian Naive Bayes with/without PCA
3. **Time-frequency decoding**: Decode from time-frequency decomposed data
4. **Permutation testing**: Assess statistical significance of decoding across subjects

## Section 1: Setup and Configuration

### 1.1 Import Required Libraries

In [ ]:
# Enable inline plotting and suppress warnings
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from IPython.display import display

# Import analysis tools
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis import BDM
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_bdm_timecourse

print("✓ All imports successful!")

### 1.2 Load Preprocessed Data with Eye-Tracking Quality Control

In [ ]:
# Download (if not already cached) and locate the preprocessed tutorial
# dataset -- a fast-path that skips running 01_preprocessing.ipynb yourself
from open_dvm.support.datasets import fetch_processed_data

project_folder = fetch_processed_data()
os.chdir(project_folder)

# Subject number (1-7 in this dataset)
sj = 2

# Eye-tracking quality control
eye_dict = {
    'use_tracker': True,        # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),      # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,          # Threshold: 1 degree visual angle
    'viewing_dist': 70,         # Viewing distance (cm)
    'screen_res': (1920, 1080), # Screen resolution (pixels)
    'screen_h': 29,             # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift correction window
}

# Load preprocessed data
df, epochs = FolderStructure().load_processed_epochs(
    sj=sj,                       # subject ID
    fname='ses_01_main',         # preprocessed file name (sub_<sj>_ses_01_main-epo.fif)
    preproc_name='main',         # preprocessing pipeline name (locates parameter files)
    eye_dict=eye_dict            # eye-tracking exclusion criteria
)

print(f'✓ Subject {sj} data loaded')
print(f'  • {len(epochs)} trials')
print(f'  • {epochs.info["nchan"]} channels')
print(f'  • Sampling rate: {epochs.info["sfreq"]} Hz')

---

## Section 2: Generalization Across Time (GAT)

**Question**: If we train a decoder at one time point, does it generalize to other time points? This reveals whether neural representations are stable or dynamic across the analysis window.

**Approach**: Train at each time point, test at all other time points. Result is a 2D matrix (train time × test time).

In [ ]:
# Initialize BDM for GAT analysis
bdm_gat = BDM(
    sj=sj,
    epochs=epochs,
    df=df,
    to_decode='dist_img',   # Decode image identity
    baseline=(-0.2, 0),     # Baseline correction: -200 to 0 ms
    nr_folds=10,            # 10-fold cross-validation
    elec_oi='all',          # Use all electrodes
    data_type='broadband',  # Time domain (not TF)
    downsample=64           # Downsample to 64 Hz for faster GAT computation
)

print(f"✓ BDM initialized for GAT analysis")

In [ ]:
# Run GAT analysis
output_gat, _ = bdm_gat.classify(
    cnds=dict(block_type=['localizer']),  # Select localizer trials
    window_oi=(-0.1, 0.4),  # Shrink window: -100 to 400 ms (500 ms total, was 700 ms)
    labels_oi='all',        # Use all distractor images
    GAT=True,               # Enable generalization across time
)

print("✓ GAT decoding complete")
print(f"  • GAT matrix shape: {output_gat['localizer']['dec_scores'].shape}")

In [ ]:
# Visualize GAT matrix
plt.figure(figsize=(7, 6))
plot_bdm_timecourse(
    output_gat,
    timecourse='2d_GAT',    # Generalization matrix (train time x test time)
    diverging_cmap=False,   # Sequential colormap (AUC scale, not a difference)
    stats=False             # Single subject -- no group-level test to run
)
plt.title('Generalization Across Time (GAT)')
plt.tight_layout()
display(plt.gcf())
plt.close()

print("✓ GAT visualization complete")

---

## Section 3: Classifier Comparisons with PCA

**Question**: How sensitive is decoding performance to classifier choice and dimensionality reduction?

**Approach**: Compare LDA, SVM, and Gaussian Naive Bayes with/without PCA preprocessing.

In [ ]:
# Classifier Comparisons with PCA
classifiers = ['LDA', 'svm', 'GNB']
pca_levels = [
    (0, 'across'),      # No PCA
    (0.95, 'across'),   # 95% variance
    (0.99, 'across')    # 99% variance
]
pca_labels = ['No PCA', '95%', '99%']
colors = ['red', 'green', 'blue']

# Initialize BDM once (will change classifier and pca_components in loops)
bdm_clf = BDM(
    sj=sj,
    epochs=epochs,
    df=df,
    to_decode='dist_img',   # Decode image identity
    baseline=(-0.2, 0),     # Baseline correction: -200 to 0 ms
    nr_folds=10,            # 10-fold cross-validation
    elec_oi='all',          # Use all electrodes
    data_type='broadband',  # Time domain (not TF)
    downsample=128,         # Downsample to 128 Hz (reduce computational load)
    scale=True              # Standardize data before PCA
)

# Create 1x3 subplots for the three classifiers
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for clf_idx, classifier in enumerate(classifiers):
    print(f'\nDecoding with {classifier} classifier...')

    # Update classifier
    bdm_clf.classifier = classifier

    # Collect outputs for all PCA levels into a single dict
    combined_output = {'info': None}

    for pca_idx, pca_comp in enumerate(pca_levels):
        # Update PCA setting
        bdm_clf.pca_components = pca_comp

        print(f'   PCA: {pca_labels[pca_idx]}', end='... ')

        # Run decoding (no GAT - just standard within-time)
        output, _ = bdm_clf.classify(
            cnds=dict(block_type=['localizer']),  # Select localizer trials
            window_oi=(-0.2, 0.5),                # Time window: -200 to 500 ms
            labels_oi='all',                      # Use all distractor images
            GAT=False,                            # Standard within-time decoding
        )

        print(f'AUC: {np.mean(output["localizer"]["dec_scores"]):.3f}')

        # Store in combined output dict
        combined_output[pca_labels[pca_idx]] = output['localizer']
        if combined_output['info'] is None:
            combined_output['info'] = output['info']

    # Plot all PCA levels for this classifier on its subplot
    plt.sca(axes[clf_idx])
    plot_bdm_timecourse(
        combined_output,
        timecourse='1d',   # 1D timecourse (no GAT)
        colors=colors,     # No PCA, 95%, 99% (see pca_labels above)
        chance_level=0.5,  # Chance level for AUC
        stats=False        # Single subject -- no group-level test to run
    )
    axes[clf_idx].set_title(f'{classifier}')
    axes[clf_idx].set_ylim([0.45, 0.75])  # Enforce consistent y-axis limits

plt.tight_layout()
display(fig)
plt.close()

print("\n✓ Classifier and PCA comparison complete")

---

## Section 4: Time-Frequency Decoding

**Question**: Can we decode distractor image identity from time-frequency representations? Which frequencies are most informative?

**Approach**: Apply time-frequency decomposition (4–40 Hz) and run decoding in frequency domain. Visualize where decoding is strongest across the time-frequency plane.

In [ ]:
# Initialize BDM with time-frequency decomposition
bdm_tfr = BDM(
    sj=sj,
    epochs=epochs,
    df=df,
    to_decode='dist_img',           # Decode distractor image identity
    baseline=(-0.2, 0),             # Baseline: -200 to 0 ms (percent change normalization)
    nr_folds=10,                    # 10-fold cross-validation
    elec_oi='all',                  # All electrodes
    data_type='tfr',                # TIME-FREQUENCY domain (not broadband time domain)
    min_freq=4, max_freq=40,        # Frequency range: theta to low gamma (4-40 Hz)
    downsample=128                  # Downsample to 128 Hz
)

print("✓ BDM initialized for TFR analysis")

In [ ]:
# Run TFR decoding on localizer task
output_tfr, _ = bdm_tfr.classify(
    cnds=dict(block_type=['localizer']),  # Select localizer task
    window_oi=(-0.2, 0.5),                # Time window: -200 to 500 ms
    labels_oi='all',                      # All distractor images
    GAT=False,                            # Standard within-time decoding (no temporal generalization)
    excl_factor=dict(img_loc=[8]),        # Exclude no-show trials
)

print("✓ TFR decoding complete")
print(f"  • TFR shape: {output_tfr['localizer']['dec_scores'].shape}")

In [ ]:
# Visualize time-frequency decoding performance
print('\nVisualizing time-frequency decoding results...')

plt.figure(figsize=(10, 6))
plot_bdm_timecourse(
    output_tfr,
    timecourse='2d_tfr',   # 2D time-frequency heatmap (time × frequency)
    diverging_cmap=False,  # Use sequential colormap (AUC scale)
    stats=False            # Single subject -- no group-level test to run
)
plt.title('Time-Frequency Decoding: Which Frequencies Decode Distractor Image?')
plt.tight_layout()
display(plt.gcf())
plt.close()

print('✓ Time-frequency decoding visualization complete')

---

## Section 5: Permutation Testing

**Question**: Is decoding performance significantly above chance across subjects for the localizer and main tasks, and do the two tasks differ from each other?

**Approach**: Run decoding across all subjects (1–7) for both `block_type` conditions in one call, then apply MNE's built-in permutation test to establish statistical significance -- both against chance (per condition) and between conditions (`cnd_diff`).

In [ ]:
# Run BDM decoding for all subjects (1-7)
print('\nDecoding distractor image across all subjects for permutation testing...')
for subject_id in range(1, 8):
    try:
        # Load data for this subject
        df_sj, epochs_sj = FolderStructure().load_processed_epochs(
            sj=subject_id,                # subject ID
            fname='ses_01_main',          # preprocessed file name (sub_<sj>_ses_01_main-epo.fif)
            preproc_name='main',          # preprocessing pipeline name (locates parameter files)
            eye_dict=eye_dict             # eye-tracking exclusion criteria
        )
        
        # Initialize BDM for this subject
        bdm_sj_temp = BDM(
            sj=subject_id,
            epochs=epochs_sj,
            df=df_sj,
            to_decode='dist_img',         # Decode distractor image identity
            baseline=(-0.2, 0),           # Baseline correction: -200 to 0 ms
            nr_folds=10,                  # 10-fold cross-validation
            elec_oi='all',                # All electrodes
            data_type='broadband',        # Time domain (not TF)
            downsample=128                # Downsample to 128 Hz
        )

        # Decode both localizer and main task trials in one call (see Section 3)
        output_localizer, _ = bdm_sj_temp.classify(
            cnds=dict(block_type=['localizer', 'main']),  # Select localizer and main task trials
            window_oi=(-0.2, 0.5),                # Time window: -200 to 500 ms
            labels_oi='all',                      # All distractor images
            GAT=False,                            # Standard within-time decoding
            excl_factor=dict(img_loc=[8]),        # Exclude no-show trials
            f_name='loc_vs_main'                  # Analysis identifier for saved output
        )
        
        print(f'  ✓ Subject {subject_id} complete')
    except Exception as e:
        print(f'  ✗ Subject {subject_id} failed: {str(e)}')

print('\n✓ All subjects processed')

In [ ]:
# Load aggregated decoding results across all subjects
print('\nLoading decoded data...')
bdm_perm_data = FolderStructure().read_bdm(
    bdm_folder_path=['dist_img', 'all_elecs'],  # Feature space and channels
    bdm_name='loc_vs_main',                     # From combined localizer+main analysis
    sjs='all'                                   # All subjects
)
print('✓ Data loaded')

In [ ]:
# Visualize decoding performance with permutation-based statistics
print('\nVisualizing permutation test results...')

plt.figure(figsize=(12, 5))
plot_bdm_timecourse(
    bdm_perm_data,
    cnds=['localizer', 'main'],      # Both conditions from the combined analysis above
    timecourse='1d',                 # 1D timecourse (not GAT)
    colors=['red', 'blue'],          # localizer, main
    stats='perm',                    # Test each condition against chance...
    cnd_diff=('localizer', 'main'),  # ...and test whether the two conditions differ
)
plt.title('Distractor Image Decoding: Localizer vs. Main Task (Permutation Test)')
plt.ylabel('AUC')
display(plt.gcf())
plt.close()

print('✓ Permutation testing complete')

---

## Section 6: Summary and Next Steps

You've explored advanced BDM techniques in a natural progression:

1. ✅ **GAT**: Temporal stability and generalization of neural representations (single-subject)
2. ✅ **Classifiers + PCA**: Trade-offs in model complexity and performance (single-subject)
3. ✅ **Time-Frequency**: Which frequencies best decode image identity (single-subject)
4. ✅ **Permutation Testing**: Localizer vs. main task decoding significance, and whether the two differ, across subjects (`cnd_diff`)

### Next Steps
- See `07_ctf_analysis.ipynb` for spatial channel tuning functions (inverted encoding model), a complementary approach to studying how continuous stimulus features are represented in EEG activity.